In [2]:
import os
import re
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import LETTER




### ISO 8601 to Seconds Converter

In [3]:
#Helper function to convert ISO 8601 duration format to seconds.
ISO8601_RE = re.compile(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?")

def iso8601_duration_to_seconds(d: str) -> int:
    """
    YouTube durations look like 'PT1M30S'. Convert to seconds.
    """
    m = ISO8601_RE.fullmatch(d)
    if not m:
        return 0
    h = int(m.group(1) or 0)
    mins = int(m.group(2) or 0)
    s = int(m.group(3) or 0)
    return h * 3600 + mins * 60 + s




### Tool Youtube Search

In [4]:

# Tool A: Youtube Search
def youtube_search_tool(youtube, query: str, max_results: int = 10) -> List[Dict[str, Any]]:

    # Video Duration is "short" which is a paramter in the search api so that we can filter videos which are less than 2 minutes.
    req = youtube.search().list(
        part="snippet",
        q=query,
        type="video",
        videoDuration="short",
        maxResults=max_results,
    )
    resp = req.execute()
    items = resp.get("items", [])
    out = []
    for it in items:
        vid = it["id"]["videoId"]
        title = it["snippet"]["title"]
        channel = it["snippet"]["channelTitle"]
        out.append({"video_id": vid, "title": title, "channel": channel})
    return out




### Tool Video Details (Duration)

In [5]:
# Tool B: Video Details (to get duration and verify it's short)
def youtube_video_details_tool(youtube, video_ids: List[str]) -> Dict[str, Dict[str, Any]]:
    # contentDetails.duration gives ISO 8601 duration string. :contentReference[oaicite:7]{index=7}
    req = youtube.videos().list(
        part="contentDetails,snippet",
        id=",".join(video_ids)
    )
    resp = req.execute()
    details = {}
    for it in resp.get("items", []):
        vid = it["id"]
        dur = it["contentDetails"]["duration"]
        details[vid] = {
            "duration": dur,
            "duration_seconds": iso8601_duration_to_seconds(dur),
            "title": it["snippet"]["title"],
            "channel": it["snippet"]["channelTitle"],
        }
    return details




### Tool Transcript

In [6]:
# ---------- Tool C: Transcript ----------
def transcript_tool(video_id: str, languages: Tuple[str, ...] = ("en",)) -> str:
    # youtube-transcript-api pulls transcripts/captions when available. :contentReference[oaicite:8]{index=8}
    parts = YouTubeTranscriptApi.get_transcript(video_id, languages=list(languages))
    return " ".join([p["text"] for p in parts])




### Tool Summarizer

In [7]:
# ---------- Tool D: Summarizer (placeholder) ----------
def summarize_tool(text: str) -> str:
    """
    Replace this with an LLM call (OpenAI, etc.) or a local summarizer.
    For now: simple truncation-based placeholder.
    """
    text = re.sub(r"\s+", " ", text).strip()
    return text[:900] + ("..." if len(text) > 900 else "")




### Tool PDF Writer

In [8]:
# ---------- Tool E: PDF writer ----------
def pdf_tool(output_path: str, sections: List[Dict[str, str]]) -> str:
    # Basic ReportLab canvas usage. :contentReference[oaicite:9]{index=9}
    c = canvas.Canvas(output_path, pagesize=LETTER)
    width, height = LETTER
    y = height - 50

    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, y, "YouTube Transcript Summaries")
    y -= 30

    c.setFont("Helvetica", 10)

    for sec in sections:
        lines = [
            f"Title: {sec['title']}",
            f"Channel: {sec['channel']}",
            f"Video ID: {sec['video_id']}",
            "",
            "Summary:",
            sec["summary"],
            "",
            "-" * 90,
            ""
        ]

        for line in lines:
            # simple wrapping
            for chunk in [line[i:i+110] for i in range(0, len(line), 110)]:
                if y < 60:
                    c.showPage()
                    c.setFont("Helvetica", 10)
                    y = height - 50
                c.drawString(50, y, chunk)
                y -= 14

    c.save()
    return output_path




In [9]:
# ---------- Agent State ----------
@dataclass
class AgentState:
    query: str
    max_seconds: int = 120
    need: int = 2
    candidates: List[Dict[str, Any]] = field(default_factory=list)
    chosen: List[Dict[str, Any]] = field(default_factory=list)
    observations: List[str] = field(default_factory=list)


# ---------- ReAct Agent ----------
def run_agent(query: str, output_pdf: str = "youtube_summaries.pdf") -> str:
    api_key = os.environ.get("YOUTUBE_API_KEY")
    if not api_key:
        raise RuntimeError("Set env var YOUTUBE_API_KEY with your YouTube Data API key.")

    youtube = build("youtube", "v3", developerKey=api_key)

    state = AgentState(query=query)

    # REASON: search
    state.observations.append(f"Searching YouTube for short videos about: {query}")
    state.candidates = youtube_search_tool(youtube, query, max_results=12)

    # REASON: verify durations + get transcripts until we have top 2 that work
    ids = [c["video_id"] for c in state.candidates]
    details = youtube_video_details_tool(youtube, ids)

    for cand in state.candidates:
        if len(state.chosen) >= state.need:
            break

        vid = cand["video_id"]
        det = details.get(vid)
        if not det:
            continue

        # OBSERVE duration
        sec = det["duration_seconds"]
        if sec <= 0 or sec > state.max_seconds:
            state.observations.append(f"Skip {vid}: duration {sec}s (needs <={state.max_seconds}s)")
            continue

        # ACT: transcript
        try:
            transcript = transcript_tool(vid)
        except (TranscriptsDisabled, NoTranscriptFound):
            state.observations.append(f"Skip {vid}: no transcript available")
            continue

        # ACT: summarize
        summary = summarize_tool(transcript)

        state.chosen.append({
            "video_id": vid,
            "title": det["title"],
            "channel": det["channel"],
            "duration_seconds": sec,
            "transcript": transcript,
            "summary": summary,
        })
        state.observations.append(f"Chosen {vid}: {sec}s, transcript ok")

    if len(state.chosen) < state.need:
        raise RuntimeError(f"Only found {len(state.chosen)} videos with <=2min duration and transcripts.")

    # ACT: write PDF
    pdf_sections = [
        {
            "video_id": x["video_id"],
            "title": x["title"],
            "channel": x["channel"],
            "summary": x["summary"],
        }
        for x in state.chosen
    ]
    return pdf_tool(output_pdf, pdf_sections)


if __name__ == "__main__":
    path = run_agent("python data science")
    print("Saved:", path)

RuntimeError: Set env var YOUTUBE_API_KEY with your YouTube Data API key.